# 한국어 BERT 뉴스 주제 분류 Fine-Tuning

**모델**: `klue/bert-base`  
**데이터셋**: `klue/ynat` (연합뉴스 주제 분류, 7개 카테고리)  
**환경**: Google Colab (GPU)

## 1. 패키지 설치

In [1]:
!pip install -q --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.3 MB/s eta 0:00:00


## 2. 라이브러리 임포트 및 GPU 확인

In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## 3. 데이터셋 로드 및 탐색

In [4]:
dataset = load_dataset("yangwang825/klue-ynat")
print(dataset)
print(f"\n컬럼: {dataset['train'].column_names}")
print(f"\n샘플 데이터:")
for i in range(3):
    print(f"  [{i}] {dataset['train'][i]}")

README.md:   0%|          | 0.00/372 [00:00<?, ?B/s]

train.json:   0%|          | 0.00/8.88M [00:00<?, ?B/s]

dev.json:   0%|          | 0.00/1.78M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 45678
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 9107
    })
})

컬럼: ['text', 'label']

샘플 데이터:
  [0] {'text': '유튜브 내달 2일까지 크리에이터 지원 공간 운영', 'label': 3}
  [1] {'text': '어버이날 맑다가 흐려져…남부지방 옅은 황사', 'label': 3}
  [2] {'text': '내년부터 국가RD 평가 때 논문건수는 반영 않는다', 'label': 2}


In [5]:
# KLUE-YNAT 컬럼 설정
label_column = "label"
text_column = "text"

# 레이블 이름 매핑
ynat_label_names = {0: "IT과학", 1: "경제", 2: "사회", 3: "생활문화", 4: "세계", 5: "스포츠", 6: "정치"}

# 레이블 분포
from collections import Counter
train_labels = dataset["train"][label_column]
counter = Counter(train_labels)
print(f"레이블 종류 수: {len(counter)}")
for k, v in sorted(counter.items()):
    print(f"  {k} ({ynat_label_names[k]}): {v}개")

레이블 종류 수: 7
  0 (IT과학): 5235개
  1 (경제): 6118개
  2 (사회): 5133개
  3 (생활문화): 5751개
  4 (세계): 8320개
  5 (스포츠): 7742개
  6 (정치): 7379개


## 4. 레이블 인코딩 및 Train/Validation/Test 분할

In [6]:
# KLUE-YNAT은 이미 정수 레이블 (0~6)
num_labels = 7
label2id = {name: idx for idx, name in ynat_label_names.items()}
id2label = {idx: name for idx, name in ynat_label_names.items()}

print(f"레이블 매핑 (총 {num_labels}개):")
for idx, name in id2label.items():
    print(f"  {idx} → {name}")

# KLUE-YNAT은 train/validation 제공 (test는 레이블 없음)
# validation을 반으로 나눠서 val/test로 사용(일반적으로는 0.7~0.8로 사용함)
# 라벨이 균형 잡히게 validation 셋을 구성하는 것이 중요함
val_test = dataset["validation"].train_test_split(test_size=0.5, seed=42, stratify_by_column="label")
train_dataset = dataset["train"]
val_dataset = val_test["train"]
test_dataset = val_test["test"]

print(f"\nTrain: {len(train_dataset)}개")
print(f"Validation: {len(val_dataset)}개")
print(f"Test: {len(test_dataset)}개")

레이블 매핑 (총 7개):
  0 → IT과학
  1 → 경제
  2 → 사회
  3 → 생활문화
  4 → 세계
  5 → 스포츠
  6 → 정치

Train: 45678개
Validation: 4553개
Test: 4554개


## 5. 토크나이저 로드 및 토큰화

In [7]:
MODEL_NAME = "klue/bert-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(examples):
    return tokenizer(
        examples[text_column],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_dataset = train_dataset.map(tokenize_fn, batched=True)
val_dataset = val_dataset.map(tokenize_fn, batched=True)
test_dataset = test_dataset.map(tokenize_fn, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

print(f"토큰화 완료 (max_length={MAX_LENGTH})")
print(f"샘플 토큰: {tokenizer.decode(train_dataset[0]['input_ids'][:30])}")

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/248k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/495k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/45678 [00:00<?, ? examples/s]

Map:   0%|          | 0/4553 [00:00<?, ? examples/s]

Map:   0%|          | 0/4554 [00:00<?, ? examples/s]

토큰화 완료 (max_length=128)
샘플 토큰: [CLS] 유튜브 내달 2일까지 크리에이터 지원 공간 운영 [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


## 6. 모델 로드

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model.to(device)
print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


모델 파라미터 수: 110,622,727


## 7. 학습 설정 및 Fine-Tuning

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1} # 정확도: 전체 정확도, F1: 라벨별 정확도를 가중치로 반영

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5, # 소수점5개
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("학습 시작...")
train_result = trainer.train()
print(f"\n학습 완료!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


학습 시작...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.351732,0.377127,0.864924,0.865015
2,0.282187,0.373092,0.867779,0.868416
3,0.201690,0.390873,0.874149,0.874276


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


학습 완료!
  Total steps: 4284
  Training loss: 0.3421


## 8. 학습 곡선 시각화

In [ ]:
log_history = trainer.state.log_history

train_losses = [(x["step"], x["loss"]) for x in log_history if "loss" in x and "eval_loss" not in x]
eval_entries = [x for x in log_history if "eval_loss" in x]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training Loss
steps, losses = zip(*train_losses)
axes[0].plot(steps, losses, label="Train Loss", color="tab:blue")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Eval Loss
eval_epochs = [x["epoch"] for x in eval_entries]
eval_losses = [x["eval_loss"] for x in eval_entries]
axes[1].plot(eval_epochs, eval_losses, marker="o", label="Eval Loss", color="tab:orange")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Validation Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Eval Metrics
eval_acc = [x["eval_accuracy"] for x in eval_entries]
eval_f1 = [x["eval_f1"] for x in eval_entries]
axes[2].plot(eval_epochs, eval_acc, marker="o", label="Accuracy", color="tab:green")
axes[2].plot(eval_epochs, eval_f1, marker="s", label="F1 (weighted)", color="tab:red")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Score")
axes[2].set_title("Validation Metrics")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. 테스트셋 평가

In [ ]:
test_results = trainer.evaluate(test_dataset)
print("=== 테스트셋 결과 ===")
print(f"  Loss:     {test_results['eval_loss']:.4f}")
print(f"  Accuracy: {test_results['eval_accuracy']:.4f}")
print(f"  F1:       {test_results['eval_f1']:.4f}")

## 10. Classification Report & Confusion Matrix

In [ ]:
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

target_names = [id2label[i] for i in range(num_labels)]
print(classification_report(labels, preds, target_names=target_names))

In [ ]:
# Confusion Matrix 시각화
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. 추론 테스트

In [ ]:
def predict_topic(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0]
    pred_id = probs.argmax().item()
    confidence = probs[pred_id].item()

    return id2label[pred_id], confidence

test_texts = [
    "삼성전자, 반도체 사업 실적 개선 기대감에 주가 상승",
    "손흥민, 프리미어리그 시즌 최다 골 기록 경신",
    "정부, 부동산 규제 완화 방안 발표 예정",
    "애플, 새로운 AI 기능 탑재한 아이폰 공개",
]

print("=== 추론 테스트 ===")
for text in test_texts:
    topic, conf = predict_topic(text)
    print(f"\n입력: {text}")
    print(f"예측: {topic} (확률: {conf:.2%})")